In [2]:
import os
import csv
import re
import yaml
import pandas as pd

In [19]:
# Path to the CSV file and output directory
date = '20260109'

vault_dir = r"C:\Users\Giorgio\Documents\GitHub\jrc-egd\Obsidian\EGD_map_v3"

output_dir = rf"C:\Users\Giorgio\Documents\GitHub\jrc-egd\LLM\Code\other_tools\obsidian_to_csv\{date}"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)



##### Extract notes links from obsidian vault

In [ ]:

# (chatGPT suggested code starts here)

# Define the output CSV file path for later use
OUTPUT_CSV = os.path.join(output_dir, "obsidian_network.csv")
VAULT_PATH = vault_dir
LINK_PATTERN = re.compile(r"\[\[([^\]]+)\]\]")


def extract_frontmatter(text):
    """Extract YAML frontmatter if present."""
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) >= 3:
            return parts[1]
    return None

def extract_links_from_value(value):
    """Extract [[links]] from any YAML value."""
    links = []

    if isinstance(value, str):
        links += LINK_PATTERN.findall(value)
    elif isinstance(value, list):
        for item in value:
            links += extract_links_from_value(item)
    elif isinstance(value, dict):
        for v in value.values():
            links += extract_links_from_value(v)

    return links

# Loop through markdown files in the vault and extract links

rows = []

for root, dirs, files in os.walk(VAULT_PATH):
    for filename in files:
        if not filename.endswith(".md"):
            continue

        path = os.path.join(root, filename)
        note_title = os.path.splitext(filename)[0]

        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        frontmatter = extract_frontmatter(text)
        links = []

        if frontmatter:
            try:
                data = yaml.safe_load(frontmatter)
                if isinstance(data, dict):
                    links = extract_links_from_value(data)
            except yaml.YAMLError:
                print(f"Warning: YAML error in {path}")

        cleaned_links = []
        for l in links:
            l = l.split("|")[0]
            l = l.split("#")[0]
            cleaned_links.append(l)

        if cleaned_links:
            for target in cleaned_links:
                rows.append((note_title, target))
        else:
            rows.append((note_title, ""))


# Write CSV
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "target"])
    writer.writerows(rows)

print(f"Exported {len(rows)} rows to {OUTPUT_CSV}")

##### Add target contant and subthemes to newly generated intra-TA link table

In [ ]:
# intra TA network made by hand 
intraTA_data = pd.read_excel(r"C:\Users\Giorgio\switchdrive\Private\JRC\network analysis\my analysis\intra_TA_links.xlsx")

# target data 150 = only targets that have been assessed in report 1
all_targets_data = pd.read_excel(r"C:\Users\Giorgio\Documents\GitHub\jrc-egd\LLM\Data\targets_data_150_xslx.xlsx")


In [22]:
# add subthemes based on

for i in range(len(intraTA_data)): 
    match = all_targets_data.loc[all_targets_data['target_code'] == intraTA_data.loc[i, 'source_target'], 'sub_theme'] 
    if not match.empty:
        intraTA_data.loc[i, 'source_target_subtheme'] = match.iloc[0]

    if 'impact_target' in intraTA_data.columns:
        match2 = all_targets_data.loc[all_targets_data['target_code'] == intraTA_data.loc[i, 'impact_target'], 'sub_theme']
        if not match2.empty:
            intraTA_data.loc[i, 'impact_target_subtheme'] = match2.iloc[0]


# add target content

for i in range(len(intraTA_data)): 
    match = all_targets_data.loc[all_targets_data['target_code'] == intraTA_data.loc[i, 'source_target'], 'target_content'] 
    if not match.empty:
        intraTA_data.loc[i, 'source_target_content'] = match.iloc[0]

    if 'impact_target' in intraTA_data.columns:
        match2 = all_targets_data.loc[all_targets_data['target_code'] == intraTA_data.loc[i, 'impact_target'], 'target_content']
        if not match2.empty:
            intraTA_data.loc[i, 'impact_target_content'] = match2.iloc[0]

# export to csv
intraTA_data.to_excel(excel_writer= f'{output_dir}\intraTA_aggregated.xlsx', index=False)


##### Extra

In [4]:
# Path to the CSV file and output directory


output_dir = rf"C:\Users\Giorgio\switchdrive\Private\JRC\network analysis\my analysis"


# target data 150 = only targets that have been assessed in report 1
all_targets_data = pd.read_excel(r"C:\Users\Giorgio\Documents\GitHub\jrc-egd\LLM\Data\targets_data_150_xslx.xlsx")

# all links corrected
all_links_corrected = pd.read_excel(r"C:\Users\Giorgio\switchdrive\Private\JRC\network analysis\my analysis\all_links_corrected.xlsx")


# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)


In [6]:
# add thematic areas based on target data 150

for i in range(len(all_links_corrected)): 
    match = all_targets_data.loc[all_targets_data['target_code'] == all_links_corrected.loc[i, 'source_target'], 'thematic_area_code'] 
    if not match.empty:
        all_links_corrected.loc[i, 'source_thematic_area'] = match.iloc[0]

    if 'impact_target' in all_links_corrected.columns:
        match2 = all_targets_data.loc[all_targets_data['target_code'] == all_links_corrected.loc[i, 'impact_target'], 'thematic_area_code']
        if not match2.empty:
            all_links_corrected.loc[i, 'impact_thematic_area'] = match2.iloc[0]


# export to excel
all_links_corrected.to_excel(excel_writer=os.path.join(output_dir, 'all_links_corrected_2.xlsx'), index=False)
